# Model Watermarking and Model Watermark Detection

This notebook demonstrates how to set up a Hugging Face Transformers model (OPT-350M) and load a text dataset using the `allenai/c4` library.

## Authors
Dong Liang (dl2287) and Mya Bridgeforth (mjb555)

## Brief Summary of the Chosen Paper
The paper proposes an algorithm for embedding a watermark in the output of large language models (LLMs) and a statistical method for detecting that watermark later. The watermark biases the model toward selecting tokens from a dynamically generated “green list” of tokens. Over long sequences, this bias produces a statistical signal that can be detected without requiring access to the original model.

### Watermark Generation Process
1.  The model tokenizes the input text.
2.  Each token is mapped to its corresponding token ID.
3.  Using a random seed derived from a secret key, the vocabulary IDs are partitioned into a green list and a red list. The green list typically contains about 25% of the vocabulary.
4.  The model produces logits representing the likelihood of each token appearing next.
5.  If a token belongs to the green list, a bias is added to its logit, increasing the probability that it will be selected.
6.  The logits are converted into probabilities using the softmax function.
7.  The next token is sampled and appended to the sequence. This process repeats until the generation is complete.

### Watermark Detection Process
1.  The green list fraction is determined by the parameter γ.
2.  Convert the input text into tokens and map each token to its corresponding token ID.
3.  Reconstruct the green list sequentially using the secret key, the previous token, and γ.
4.  If the current token belongs to the reconstructed green list, increment greenCount.
5.  Repeat this process for every token in the sequence.
6.  Compute the following z-score to determine whether the text is watermarked:

    z = (G − γT) / sqrt(Tγ(1 − γ))
    
    where:
    *   G = number of observed green tokens (greenCount)
    *   T = total number of tokens
    *   γ = fraction of the vocabulary assigned to the green list
7.  In the paper, a watermark is considered strongly detected when z ≥ 4.

## Sections:

1.  **Install Necessary Libraries**
2.  **Model and Tokenizer**
3.  **Dataset Loading and Cleaning**
4.  **Green/Red List**

## 1. Install Necessary Libraries

We'll install the `transformers` and `torch` libraries, which are essential for working with pre-trained language models and deep learning operations.


In [ ]:
!pip install transformers torch

## 2. Model and Tokenizer

We load the `facebook/opt-350m` model and its corresponding tokenizer. This model is a causal language model, suitable for text generation tasks. We also detect if a CUDA-enabled GPU is available to utilize it; otherwise, the CPU will be used.

*Note: we are using AutoModel version for more control over the model


In [ ]:
#@title
import torch
import math
import hashlib
import random
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from transformers import AutoTokenizer, AutoModelForCausalLM, AutoModelForSeq2SeqLM
from datasets import load_dataset
from itertools import islice
from torch.nn.utils.rnn import pad_sequence
from matplotlib.colors import ListedColormap, BoundaryNorm
from matplotlib.lines import Line2D

In [ ]:
# @title
device = "cuda" if torch.cuda.is_available() else "cpu"
tokenizer = AutoTokenizer.from_pretrained("facebook/opt-350m")
model = AutoModelForCausalLM.from_pretrained("facebook/opt-350m").to(device)
model.eval()

## 3. Dataset Loading and Cleaning

Here, we load a large English text dataset (`allenai/c4`) using the `datasets` library in streaming mode. This allows us to process data efficiently without loading the entire dataset into memory.

After loading our dataset, we have decided to keep texts where 250 <= text_size <= 450. This is then followed by turning them into 10 batches of size 32 where each tokenized text is padded to length 450.

Since we are padding in order to do batch processing, we need an attention mask in order to tell which part of the data are actual tokens and not just padded junks.

In [ ]:
english_texts = load_dataset("allenai/c4", "en", streaming=True)

In [ ]:
MIN_TOKEN_LENGTH = 250
MAX_TOKEN_LENGTH = 400

TEST_BATCH_SIZE = 5
TEST_NUM_BATCHES = 5

REAL_BATCH_SIZE = 10
REAL_NUM_BATCHES = 30

stream = iter(english_texts["train"])

In [ ]:
def get_clean_text_batch(stream, tokenizer, batch_size, MIN_TOKEN_LENGTH=250, MAX_TOKEN_LENGTH=450):
    """
    This function returns a cleaned text batch where the each text's token length is between 250 and 450.
    """
    texts = []

    while len(texts) < batch_size:
        item = next(stream)
        text = item["text"].strip()

        if len(text) == 0:
            continue

        tokens = tokenizer(text, return_tensors="pt").input_ids
        if tokens.shape[1] < 250 or tokens.shape[1] > 450:
            continue

        texts.append(text)

    return texts

In [ ]:
def get_all_tokenized_batch(stream, tokenizer, num_batches, batch_size, MIN_TOKEN_LENGTH, MAX_TOKEN_LENGTH, pad_size=450):
    """
    This function returns a list of tokenized batches.
    """
    all_batches = []

    for _ in range(num_batches):
        # texts is the actual texts for one single batch
        texts = get_clean_text_batch(stream, tokenizer, batch_size, MIN_TOKEN_LENGTH, MAX_TOKEN_LENGTH)

        # tokenize that batch
        tokenized = tokenizer(
            texts,
            return_tensors="pt",
            padding="max_length",
            max_length=MAX_TOKEN_LENGTH,
            truncation=True
        )

        input_ids = tokenized["input_ids"]
        attention_mask = tokenized["attention_mask"]

        all_batches.append((input_ids, attention_mask))

    return all_batches

# test data
test_batches = get_all_tokenized_batch(stream, tokenizer, TEST_NUM_BATCHES, TEST_BATCH_SIZE, MIN_TOKEN_LENGTH, MAX_TOKEN_LENGTH)

# real data (for graph and final result)
real_batches = get_all_tokenized_batch(stream, tokenizer, REAL_NUM_BATCHES, REAL_BATCH_SIZE, MIN_TOKEN_LENGTH, MAX_TOKEN_LENGTH)

In [ ]:
# Note that batches[x] is a tuple (input_ids, attention_mask)
# input_ids is the actual token_ids
# attention_mask is an array that tells which part of the input_ids are actual tokens and are not padded junks

print(f"Number of batches in test_batches: {len(test_batches)}")
print(f"test_batch batch size: {len(test_batches[0][0])}")

print(f"Number of batches in real_batches: {len(real_batches)}")
print(f"real_batch batch size: {len(real_batches[0][0])}")

## 4. Green/Red List

We will now partition all tokens to either green or red list where green list will contain about 25% of the vocabulary. To generate the green list list, we will need:
- `prev_token_id`: the previous token's id
- `vocab_size`: size of the vocab in our `tokenizer`
- `key`: a randomly generated number so we can reproduce our green list
- `gamma`(optional): percentage of size of green_list to full vocab. Defaults to 0.25

In [ ]:
def get_green_list(prev_token_id, vocab_size, key, gamma=0.1):
    seed = (prev_token_id + key) % (2**32)
    rng = np.random.default_rng(seed)

    perm = rng.permutation(vocab_size)

    k = int(gamma * vocab_size)

    return torch.tensor(perm[:k], device=device)

## 5. Watermark Generation
max_new_tokens: max number of token to generate

delta (δ): amount to add to the logits of each green list token

gamma(optional): percentage of size of green_list to full vocab. Defaults to 0.25

In [ ]:
def generate(input_ids, attention_mask, max_new_tokens=200, gamma=0.25, delta=1.0, key=13):
    model.eval()

    eos_token_id = tokenizer.eos_token_id
    pad_token_id = tokenizer.pad_token_id
    if pad_token_id is None:
        pad_token_id = eos_token_id

    input_ids = input_ids.to(device)
    attention_mask = attention_mask.to(device)

    batch_size = input_ids.size(0)

    generated_ids = torch.full(
        (batch_size, max_new_tokens),
        pad_token_id,
        dtype=input_ids.dtype,
        device=device
    )

    generated_mask = torch.zeros(
        (batch_size, max_new_tokens),
        dtype=attention_mask.dtype,
        device=device
    )

    unfinished = torch.ones(batch_size, dtype=torch.bool, device=device)

    past_key_values = None

    for step in range(max_new_tokens):
        with torch.no_grad():
            if step == 0:
                output = model(
                    input_ids=input_ids,
                    attention_mask=attention_mask,
                    use_cache=True
                )
                logits = output.logits
                last_indices = attention_mask.sum(dim=1) - 1
            else:
                output = model(
                    input_ids=next_tokens,
                    past_key_values=past_key_values,
                    use_cache=True
                )
                logits = output.logits

        past_key_values = output.past_key_values

        next_tokens_list = []

        for i in range(batch_size):
            if not unfinished[i]:
                next_token = torch.tensor([pad_token_id], device=device)
                next_tokens_list.append(next_token)
                continue

            if step == 0:
                prev_token_id = input_ids[i, last_indices[i]].item()
                logits_i = logits[i, last_indices[i], :].clone()
            else:
                prev_token_id = input_ids[i, -1].item()
                logits_i = logits[i, -1, :].clone()

            vocab_size = tokenizer.vocab_size

            green_list = get_green_list(
                prev_token_id,
                vocab_size,
                key,
                gamma
            )

            logits_i[green_list] += delta

            probs = torch.softmax(logits_i, dim=-1)

            next_token = torch.multinomial(probs, num_samples=1)

            next_tokens_list.append(next_token)

            generated_ids[i, step] = next_token.item()
            generated_mask[i, step] = 1

            if next_token.item() == eos_token_id:
                unfinished[i] = False

        next_tokens = torch.stack(next_tokens_list, dim=0)

        input_ids = torch.cat([input_ids, next_tokens], dim=1)

        new_mask = unfinished.unsqueeze(1).to(attention_mask.dtype)
        attention_mask = torch.cat([attention_mask, new_mask], dim=1)

        if not unfinished.any():
            break

    return generated_ids, generated_mask

## 6. Watermark Detection
G = number of green list token found

gamma = percentage of green list to full vocab_size

T = total unpadded/valid tokens

We need our original input_tokenized_batch (test/real_batches) because first green_list must come from the last tokenized_text instead of the first tokenized_text of the generated text.

In [ ]:
def compute_z_scores(generated_tokenized_batch, generated_attention_mask, vocab_size, gamma=0.25, key=13):
    z_scores = []

    batch_size = generated_tokenized_batch.shape[0]

    for i in range(batch_size):
        generated_tokenized = generated_tokenized_batch[i]

        G = 0
        T = 0  # number of tokens actually checked

        for j in range(1, generated_tokenized.shape[0]):

            if generated_attention_mask[i, j] == 0:
                break

            prev_token = generated_tokenized[j - 1].item()
            current_token = generated_tokenized[j].item()

            green_list = get_green_list(prev_token, vocab_size, key, gamma)

            if current_token in green_list:
                G += 1

            T += 1

        denominator = math.sqrt(T * gamma * (1 - gamma))

        if denominator <= 0:
            z_scores.append(0.0)
        else:
            z = (G - gamma * T) / denominator
            z_scores.append(z)

    return z_scores



def detect_watermark(input_ids, attention_mask, vocab_size, gamma=0.25, key=13, threshold=4.0):
    z = compute_z_scores(input_ids, attention_mask, vocab_size, gamma, key)

    z_tensor = torch.tensor(z)

    strong_watermark = z_tensor >= threshold

    print(f"This batch has {strong_watermark.sum().item()} strong watermarks out of {len(z)}")

    return z_tensor, strong_watermark

## 7. Test Run

**max_new_token:** Controls output length. Higher values produce longer text and make the watermark easier to detect.

**gamma:** Controls the fraction of favored (“green”) tokens. Lower values strengthen the watermark signal but can reduce text naturalness.

**delta:** Controls how strongly green tokens are preferred. Higher values make the watermark more detectable but can degrade text quality.

**threshold:** Controls how strict detection is. Higher values reduce false positives but may miss weaker watermarks.

In [ ]:
BASE_GAMMA = 0.25
BASE_DELTA = 2.0
BASE_THRESHOLD = 4.0
BASE_LENGTH = 100

NUM_RUNS = 20

In [ ]:
def do_test_run(max_new_token=50, gamma=0.25, delta=2.0, threshold=4.0):

    prompt = test_batches

    for input_ids, attention_mask in prompt:

      watermarked_output, watermarked_attention_mask = generate(
          input_ids,
          attention_mask,
          max_new_token,
          gamma,
          delta,
          key=13
      )
      detect_watermark(
          watermarked_output,
          watermarked_attention_mask,
          vocab_size=tokenizer.vocab_size,
          gamma=gamma,
          threshold=threshold,
          key=13,
      )

In [ ]:
do_test_run()

## 8. Replicating the Paper Graphs (Figures 3a and 3b)


In [ ]:
def z_over_time(tokenized_output, output_mask, gamma, max_T=100, key=13):

    tokens = tokenized_output[0]

    zs = []
    lengths = []

    for T in range(10, max_T + 1, 10):

        # stop if we don’t have enough tokens
        if T > tokens.shape[0]:
            break

        prefix = tokens[:T].unsqueeze(0)
        prefix_mask = torch.ones_like(prefix)  # force full length

        z = compute_z_scores(
            prefix,
            prefix_mask,
            tokenizer.vocab_size,
            gamma=gamma,
            key=key
        )[0]

        zs.append(z)
        lengths.append(T)

    return np.array(lengths), np.array(zs)

In [ ]:
oracle_model_name = "facebook/opt-350m"

oracle_tokenizer = AutoTokenizer.from_pretrained(oracle_model_name)
oracle_model = AutoModelForCausalLM.from_pretrained(oracle_model_name).to(device)
oracle_model.eval()

In [ ]:
def compute_perplexity(prompt_ids, prompt_mask, gen_ids, gen_mask):
    prompt_ids = prompt_ids.to(device)
    prompt_mask = prompt_mask.to(device)
    gen_ids = gen_ids.to(device)
    gen_mask = gen_mask.to(device)

    input_ids = torch.cat([prompt_ids, gen_ids], dim=1)
    attention_mask = torch.cat([prompt_mask, gen_mask], dim=1)

    labels = input_ids.clone()

    prompt_len = prompt_ids.shape[1]

    labels[:, :prompt_len] = -100
    labels[attention_mask == 0] = -100

    with torch.no_grad():
        outputs = oracle_model(
            input_ids=input_ids,
            attention_mask=attention_mask,
            labels=labels
        )

    return torch.exp(outputs.loss).item()

In [ ]:
delta_colors = {
        10.0: "#32b77a",
        5.0: "#228e8c",
        2.0: "#32668d",
        1.0: "#443a83",
        0.5: "#440153",
    }

gamma_colors = {
        0.1: "#32b77a",
        0.25: "#228e8c",
        0.5: "#32668d",
        0.75: "#443a83",
        0.9: "#440153"
    }

In [ ]:
def plot_delta_sweep(real_batches, num_runs=NUM_RUNS):

    deltas = [10.0, 5.0, 2.0, 1.0, 0.5]
    gamma = 0.25

    prompt_ids, prompt_mask = real_batches[0]

    plt.figure(figsize=(7, 5))

    z_points = []

    for d in deltas:

        all_z = []

        for _ in range(num_runs):

            gen, gen_mask = generate(
                prompt_ids,
                prompt_mask,
                max_new_tokens=200,
                gamma=gamma,
                delta=d,
                key=13
            )

            T, z = z_over_time(gen, gen_mask, gamma=gamma, max_T=200)
            all_z.append(z)

        all_z = np.stack(all_z)
        avg_z_curve = np.mean(all_z, axis=0)

        avg_z = np.mean(avg_z_curve)

        z_points.append(avg_z)

        plt.plot(
            T,
            avg_z_curve,
            label=f"δ={d}, γ={gamma}",
            color=delta_colors[d]
        )

    plt.title("Z-score vs Token Length (varying δ, γ=0.25)")
    plt.xlabel("T (generated tokens)")
    plt.ylabel("z-score")
    plt.grid(True)
    plt.legend()
    plt.show()

In [ ]:
def plot_gamma_sweep(real_batches, num_runs=NUM_RUNS):

    gammas = [0.1, 0.25, 0.5, 0.75, 0.9]
    delta = 5.0

    prompt_ids, prompt_mask = real_batches[0]

    plt.figure(figsize=(7, 5))

    for g in gammas:
        all_z = []

        for _ in range(num_runs):

            gen, gen_mask = generate(
                prompt_ids,
                prompt_mask,
                max_new_tokens=200,
                gamma=g,
                delta=delta,
                key=13
            )

            T, z = z_over_time(
                gen,
                gen_mask,
                gamma=g,
                max_T=200,
                key=13
            )

            all_z.append(z)

        all_z = np.stack(all_z)
        avg_z = np.mean(all_z, axis=0)

        plt.plot(
            T,
            avg_z,
            label=f"δ={delta}, γ={g}",
            color=gamma_colors[g]
        )

    plt.title("Z-score vs Token Length (varying γ, δ=5.0)")
    plt.xlabel("T (generated tokens)")
    plt.ylabel("z-score")
    plt.legend()
    plt.grid(True)
    plt.show()

In [ ]:
def plot_tradeoff_surface(real_batches, num_runs=NUM_RUNS):

    prompt_ids, prompt_mask = real_batches[0]

    gammas = [0.1, 0.25, 0.5, 0.75, 0.9]
    deltas = [10.0, 5.0, 2.0, 1.0, 0.5]

    delta_markers = {
        10.0: "+",
        5.0: "*",
        2.0: "p",
        1.0: "x",
        0.5: "o"
    }

    colors = [gamma_colors[g] for g in gammas][::-1]
    cmap = ListedColormap(colors)

    boundaries = np.linspace(0, len(gammas), len(gammas) + 1)
    norm = BoundaryNorm(boundaries, cmap.N)

    points = []

    for gi, g in enumerate(gammas):
        for d in deltas:

            z_vals = []
            ppl_vals = []

            for _ in range(num_runs):

                gen, gen_mask = generate(
                    prompt_ids,
                    prompt_mask,
                    max_new_tokens=200,
                    gamma=g,
                    delta=d,
                    key=13
                )

                _, z_curve = z_over_time(gen, gen_mask, gamma=g, max_T=200)
                z_vals.append(np.mean(z_curve))

                ppl_vals.append(
                    compute_perplexity(prompt_ids, prompt_mask, gen, gen_mask)
                )

            points.append({
                "gamma": g,
                "gamma_idx": gi,
                "delta": d,
                "z": np.mean(z_vals),
                "ppl": np.mean(ppl_vals)
            })

    fig, ax = plt.subplots(figsize=(9, 6))

    ax.invert_xaxis()
    ax.grid(True)

    for p in points:
        ax.scatter(
            p["ppl"],
            p["z"],
            marker=delta_markers[p["delta"]],
            color=gamma_colors[p["gamma"]],
            s=120
        )

    ax.set_xlabel("Perplexity (lower = better)")
    ax.set_ylabel("Z-score")
    ax.set_title("Watermark Tradeoff: Z-score vs Perplexity")

    ax.axhline(y=BASE_THRESHOLD, linestyle=":")

    sm = plt.cm.ScalarMappable(cmap=cmap, norm=norm)
    sm.set_array([])

    cbar = plt.colorbar(sm, ax=ax)
    cbar.set_ticks(np.arange(len(gammas)) + 0.5)
    cbar.set_ticklabels(gammas[::-1])
    cbar.set_label("Gamma (γ)")

    delta_legend = [
        Line2D([0], [0],
               marker=delta_markers[d],
               color='black',
               linestyle='None',
               markersize=10,
               label=f"δ = {d}")
        for d in deltas
    ]

    ax.legend(
        handles=delta_legend,
        title="Delta (shape)",
        loc="upper right",
        frameon=True
    )

    plt.show()

In [ ]:
plot_gamma_sweep(real_batches)

In [ ]:
plot_delta_sweep(real_batches)

In [ ]:
plot_tradeoff_surface(real_batches)

## 9. Exetnsion: Further Analysis to Determine Optimal Parameters

We were interested in testing whether there were better parameters for the Watermark than what the paper chose, thus we experimented over delta, gamma, the threshold, and length. The latter to determine at what length the Watermark process begins to be reliable.

In [ ]:
def evaluate_detection(
    real_batches,
    gamma,
    delta,
    threshold,
    max_new_tokens,
    num_runs=NUM_RUNS,
    key=13
):
    tp = 0
    fp = 0
    total = 0

    z_w_list = []
    z_b_list = []

    for i in range(num_runs):
        input_ids, attention_mask = real_batches[i % len(real_batches)]

        # generate() returns:
        # generated_tokens: (batch_size, max_new_tokens)
        # generated_mask:   (batch_size, max_new_tokens)
        watermarked, wm_mask = generate(
            input_ids=input_ids,
            attention_mask=attention_mask,
            max_new_tokens=max_new_tokens,
            gamma=gamma,
            delta=delta,
            key=key
        )

        baseline, base_mask = generate(
            input_ids=input_ids,
            attention_mask=attention_mask,
            max_new_tokens=max_new_tokens,
            gamma=gamma,
            delta=0.0,
            key=key
        )

        z_w = compute_z_scores(
            watermarked,
            wm_mask,
            tokenizer.vocab_size,
            gamma=gamma,
            key=key
        )

        z_b = compute_z_scores(
            baseline,
            base_mask,
            tokenizer.vocab_size,
            gamma=gamma,
            key=key
        )

        z_w_list.extend(z_w)
        z_b_list.extend(z_b)

        tp += sum(z >= threshold for z in z_w)
        fp += sum(z >= threshold for z in z_b)

        total += len(z_w)

    tpr = tp / total
    fpr = fp / total
    avg_z_w = np.mean(z_w_list)
    avg_z_b = np.mean(z_b_list)

    return tpr, fpr, avg_z_w, avg_z_b

def run_and_plot_experiment(
    param_name,
    param_values,
    real_batches,
    evaluate_detection_fn,
    base_gamma=BASE_GAMMA,
    base_delta=BASE_DELTA,
    base_threshold=BASE_THRESHOLD,
    base_max_new_token=BASE_LENGTH,
    num_runs=NUM_RUNS
):

    tpr_list = []
    fpr_list = []

    for v in param_values:

        gamma = base_gamma
        delta = base_delta
        threshold = base_threshold
        max_new_token = base_max_new_token

        if param_name == "gamma":
            gamma = v
        elif param_name == "delta":
            delta = v
        elif param_name == "threshold":
            threshold = v
        elif param_name == "length":
            max_new_token = v

        tpr, fpr, avg_z_w, avg_z_b = evaluate_detection_fn(
            real_batches=real_batches,
            gamma=gamma,
            delta=delta,
            threshold=threshold,
            max_new_tokens=max_new_token,
            num_runs=num_runs
        )

        tpr_list.append(tpr)
        fpr_list.append(fpr)

    plt.figure()
    plt.plot(param_values, tpr_list, label="TPR")
    plt.plot(param_values, fpr_list, label="FPR")
    plt.xlabel(param_name.capitalize())
    plt.ylabel("Rate")
    plt.title(f"Effect of {param_name.capitalize()} on Watermark Detection")
    plt.legend()
    plt.show()

In [ ]:
gammas = [0.05, 0.1, 0.25, 0.4, 0.7, 1]

run_and_plot_experiment(
    param_name="gamma",
    param_values=gammas,
    real_batches=real_batches,
    evaluate_detection_fn=evaluate_detection
)

In [ ]:
deltas = [0.1, 0.5, 2, 3, 4]

run_and_plot_experiment(
    param_name="delta",
    param_values=deltas,
    real_batches=real_batches,
    evaluate_detection_fn=evaluate_detection
)

In [ ]:
thresholds = [-2, -1, 0.1, 0.5, 1, 2, 3, 4, 6]

run_and_plot_experiment(
    param_name="threshold",
    param_values=thresholds,
    real_batches=real_batches,
    evaluate_detection_fn=evaluate_detection,
    num_runs=5
)

In [ ]:
lengths = [20, 70, 80, 90, 100, 110, 120, 130, 140]

run_and_plot_experiment(
    param_name="length",
    param_values=lengths,
    real_batches=real_batches,
    evaluate_detection_fn=evaluate_detection
)

## 10. Extension: Effect of translation on the watermark

We were curious as to whether the Watermark would persist through a "Backtransalation Attack". Ie taking English watermarked text, translating it to another language (in this case German), then back to English. Given how translation is not necessarily deterministic, this scenario caught our attention. We varied the parameter delta to see how strong of a Watermark was needed for it to survive, and we varied length to see what effect that had on the Watermark's persistence. Would longer or shorter sequences be more vulnerable to this attack?

In [ ]:
en_model_name = "Helsinki-NLP/opus-mt-en-de"
de_model_name = "Helsinki-NLP/opus-mt-de-en"

en_tokenizer = AutoTokenizer.from_pretrained(en_model_name)
en_model = AutoModelForSeq2SeqLM.from_pretrained(en_model_name).to(device)

de_tokenizer = AutoTokenizer.from_pretrained(de_model_name)
de_model = AutoModelForSeq2SeqLM.from_pretrained(de_model_name).to(device)

In [ ]:
def translate(text, tokenizer, model, max_new_tokens):
    inputs = tokenizer(text, return_tensors="pt", truncation=True).to(device)

    with torch.no_grad():
        out = model.generate(**inputs, max_new_tokens=max_new_tokens)

    return tokenizer.decode(out[0], skip_special_tokens=True)


def backtranslate(tokenized_output, max_new_tokens):
    text = tokenizer.decode(tokenized_output[0], skip_special_tokens=True)

    german = translate(text, en_tokenizer, en_model, max_new_tokens)
    back = translate(german, de_tokenizer, de_model, max_new_tokens)

    return tokenizer(back, return_tensors="pt").input_ids.to(device)

In [ ]:
def run_experiment(
    real_batches=real_batches,
    max_new_tokens=BASE_LENGTH,
    gamma=BASE_GAMMA,
    delta=BASE_DELTA,
    num_runs=NUM_RUNS,
    key=13
):

    rows = []

    for i in range(num_runs):

        input_ids, attention_mask = real_batches[i % len(real_batches)]

        clean_ids, clean_mask = generate(
            input_ids,
            attention_mask,
            max_new_tokens=max_new_tokens,
            gamma=gamma,
            delta=0.0,
            key=key
        )

        z_clean = compute_z_scores(
            clean_ids,
            clean_mask,
            vocab_size=tokenizer.vocab_size,
            gamma=gamma,
            key=key
        )

        wm_ids, wm_mask = generate(
            input_ids,
            attention_mask,
            max_new_tokens=max_new_tokens,
            gamma=gamma,
            delta=delta,
            key=key
        )

        z_wm = compute_z_scores(
            wm_ids,
            wm_mask,
            vocab_size=tokenizer.vocab_size,
            gamma=gamma,
            key=key
        )

        bt_ids = backtranslate(wm_ids, max_new_tokens)
        bt_mask = torch.ones_like(bt_ids)

        z_bt = compute_z_scores(
            bt_ids,
            bt_mask,
            vocab_size=tokenizer.vocab_size,
            gamma=gamma,
            key=key
        )

        rows.append({
            "Run": i + 1,
            "Unwatermarked": float(np.mean(z_clean)),
            "Watermarked": float(np.mean(z_wm)),
            "Backtranslated": float(np.mean(z_bt))
        })

    return pd.DataFrame(rows)

In [ ]:
df = run_experiment(
    # max_new_tokens=50,
    # gamma=0.1,
    # delta=3.0,
    # num_runs=5
)

In [ ]:
print(df)

In [ ]:
plt.plot(df["Run"], df["Unwatermarked"], label="Unwatermarked")
plt.plot(df["Run"], df["Watermarked"], label="Watermarked")
plt.plot(df["Run"], df["Backtranslated"], label="Backtranslated Watermarked")

plt.axhline(y=BASE_THRESHOLD, linestyle=":", label=f"Detection Threshold (z={BASE_THRESHOLD})")

plt.xlabel("Run")
plt.ylabel("Z-score")
plt.legend()
plt.title("Z-score per Run")
plt.show()

In [ ]:
def run_z_over_length_experiment(
    real_batches=real_batches,
    max_new_tokens_list=[50, 100, 150, 200],
    gamma=BASE_GAMMA,
    delta=BASE_DELTA,
    num_runs=NUM_RUNS,
    key=13
):

    prompt_ids, prompt_mask = real_batches[0]
    rows = []

    for T in max_new_tokens_list:

        for run in range(num_runs):

            clean_ids, clean_mask = generate(
                prompt_ids,
                prompt_mask,
                max_new_tokens=T,
                gamma=gamma,
                delta=0.0,
                key=key
            )

            wm_ids, wm_mask = generate(
                prompt_ids,
                prompt_mask,
                max_new_tokens=T,
                gamma=gamma,
                delta=delta,
                key=key
            )

            bt_ids = backtranslate(wm_ids, T)
            bt_mask = torch.ones_like(bt_ids)

            z_clean = compute_z_scores(
                clean_ids,
                clean_mask,
                tokenizer.vocab_size,
                gamma=gamma,
                key=key
            )

            z_wm = compute_z_scores(
                wm_ids,
                wm_mask,
                tokenizer.vocab_size,
                gamma=gamma,
                key=key
            )

            z_bt = compute_z_scores(
                bt_ids,
                bt_mask,
                tokenizer.vocab_size,
                gamma=gamma,
                key=key
            )

            rows.append({
                "T": T,
                "Run": run,
                "Clean": np.mean(z_clean),
                "Watermarked": np.mean(z_wm),
                "Backtranslated": np.mean(z_bt)
            })

    return pd.DataFrame(rows)

In [ ]:
def run_delta_experiment(
    real_batches=real_batches,
    deltas=[10.0, 5.0, 2.0, 1.0, 0.5],
    gamma=BASE_GAMMA,
    max_new_tokens=BASE_LENGTH,
    num_runs=NUM_RUNS,
    key=13
):

    rows = []

    input_ids, attention_mask = real_batches[0]

    for d in deltas:

        z_clean_vals = []
        z_wm_vals = []
        z_bt_vals = []

        for _ in range(num_runs):

            clean_ids, clean_mask = generate(
                input_ids,
                attention_mask,
                max_new_tokens=max_new_tokens,
                gamma=gamma,
                delta=0.0,
                key=key
            )

            wm_ids, wm_mask = generate(
                input_ids,
                attention_mask,
                max_new_tokens=max_new_tokens,
                gamma=gamma,
                delta=d,
                key=key
            )

            bt_ids = backtranslate(wm_ids, max_new_tokens)
            bt_mask = torch.ones_like(bt_ids)

            z_clean = compute_z_scores(
                clean_ids,
                clean_mask,
                vocab_size=tokenizer.vocab_size,
                gamma=gamma,
                key=key
            )

            z_wm = compute_z_scores(
                wm_ids,
                wm_mask,
                vocab_size=tokenizer.vocab_size,
                gamma=gamma,
                key=key
            )

            z_bt = compute_z_scores(
                bt_ids,
                bt_mask,
                vocab_size=tokenizer.vocab_size,
                gamma=gamma,
                key=key
            )

            z_clean_vals.append(np.mean(z_clean))
            z_wm_vals.append(np.mean(z_wm))
            z_bt_vals.append(np.mean(z_bt))

        rows.append({
            "Delta": d,
            "Clean": float(np.mean(z_clean_vals)),
            "Watermarked": float(np.mean(z_wm_vals)),
            "Backtranslated": float(np.mean(z_bt_vals))
        })

    return pd.DataFrame(rows)

In [ ]:
length_df = run_z_over_length_experiment()

In [ ]:
grouped = length_df.groupby("T").mean(numeric_only=True).sort_index()

plt.figure(figsize=(7, 5))

plt.plot(grouped.index, grouped["Clean"], label="Clean")
plt.plot(grouped.index, grouped["Watermarked"], label="Watermarked")
plt.plot(grouped.index, grouped["Backtranslated"], label="Backtranslated")

plt.axhline(
    y=BASE_THRESHOLD,
    linestyle=":",
    label=f"Threshold (z={BASE_THRESHOLD})"
)

plt.xlabel("Sequence length")
plt.ylabel("Z-score")
plt.title("Backtranslate: Z-score vs Generation Length")
plt.grid(True)
plt.legend()
plt.show()

In [ ]:
delta_df = run_delta_experiment()

In [ ]:
plt.plot(delta_df["Delta"], delta_df["Clean"], label="Clean")
plt.plot(delta_df["Delta"], delta_df["Watermarked"], label="Watermarked")
plt.plot(delta_df["Delta"], delta_df["Backtranslated"], label="Backtranslated")

plt.axhline(
    y=BASE_THRESHOLD,
    linestyle=":",
    label=f"Threshold (z={BASE_THRESHOLD})"
)

plt.xlabel("Delta")
plt.ylabel("Z-score")
plt.title("Backtranslate: Z-score vs Delta")
plt.grid(True)
plt.legend()
plt.show()

We noticed that at delta approximately = 2, the backtranslated watermark started being detected. This was an interesting result, as that is the value the paper chose for its delta despite not conducting any bactranslation attacks. This further proves how robust the paper's parameter selection is. We also found that similar to the basic Watermark scenario, shorter sequences were harder for the Watermark to be detected/persist on.